In [ ]:
# pip install -U datasets evaluate transformers accelerate torch gradio scikit-learn matplotlib
# If you're in a notebook environment, you may prefer:
# %pip install -U datasets evaluate transformers accelerate torch gradio scikit-learn matplotlib

%matplotlib inline


# Task 1 — News Topic Classifier (BERT)

## Objective
Fine-tune **`bert-base-uncased`** on the **AG News** dataset to classify news headlines/articles into 4 categories.

## Dataset
AG News via `datasets.load_dataset("ag_news")` (public, no login).

## Approach
- Tokenize with `AutoTokenizer`
- Split train/validation (90/10)
- Fine-tune with Hugging Face `Trainer` (3 epochs on GPU; 1 epoch on CPU for a quick demo)
- Evaluate with accuracy + weighted F1
- Save the model locally
- Deploy a small Gradio demo

## Final Summary / Insights
This notebook is designed to run quickly and safely on free resources (CPU-only is OK). For better results, increase the sample size and run on a GPU.


In [ ]:
import os
import random
from dataclasses import dataclass
from typing import Any, Dict

import numpy as np

try:
    import torch
except Exception as e:
    raise ImportError("PyTorch is required. Install with: pip install torch") from e

try:
    import evaluate
except Exception as e:
    raise ImportError("evaluate is required. Install with: pip install evaluate") from e

try:
    from datasets import load_dataset
except Exception as e:
    raise ImportError("datasets is required. Install with: pip install datasets") from e

try:
    from transformers import (
        AutoModelForSequenceClassification,
        AutoTokenizer,
        DataCollatorWithPadding,
        Trainer,
        TrainingArguments,
        pipeline,
        set_seed,
    )
except Exception as e:
    raise ImportError("transformers is required. Install with: pip install transformers") from e

SEED = int(os.getenv("SEED", "42"))
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

@dataclass
class Config:
    model_name: str = "bert-base-uncased"
    max_length: int = 128
    train_samples_cpu: int = 500
    train_samples_gpu: int = 5000
    num_train_epochs_gpu: int = 3
    num_train_epochs_cpu: int = 1
    output_dir: str = "./task1_bert_agnews_model"

cfg = Config()
print(cfg)


In [ ]:
# Load AG News and create a 90/10 train/validation split
try:
    ds = load_dataset("ag_news")
except Exception as e:
    raise RuntimeError(
        "Failed to download/load AG News. Check your internet connection and try again. "
        f"Original error: {e}"
    ) from e

label_names = ds["train"].features["label"].names
print("Labels:", label_names)

splits = ds["train"].train_test_split(test_size=0.1, seed=SEED)
train_ds = splits["train"]
val_ds = splits["test"]

# Subsample for fast demo
train_limit = cfg.train_samples_gpu if device == "cuda" else cfg.train_samples_cpu
val_limit = max(100, int(train_limit * 0.2))
train_ds = train_ds.select(range(min(train_limit, len(train_ds))))
val_ds = val_ds.select(range(min(val_limit, len(val_ds))))

print("Using train samples:", len(train_ds), "val samples:", len(val_ds))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

def tokenize_batch(batch: Dict[str, Any]) -> Dict[str, Any]:
    return tokenizer(batch["text"], truncation=True, max_length=cfg.max_length)

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
val_tok = val_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in id2label.items()}

model = AutoModelForSequenceClassification.from_pretrained(
    cfg.model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)


In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1w = f1.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1_weighted": f1w}

num_train_epochs = cfg.num_train_epochs_gpu if device == "cuda" else cfg.num_train_epochs_cpu

training_args = TrainingArguments(
    output_dir="./task1_bert_runs",
    learning_rate=2e-5,
    per_device_train_batch_size=16 if device == "cuda" else 8,
    per_device_eval_batch_size=32,
    num_train_epochs=num_train_epochs,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    logging_steps=25,
    report_to="none",
    fp16=(device == "cuda"),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
metrics = trainer.evaluate()
print("Final metrics:", metrics)


In [ ]:
# Save model locally
os.makedirs(cfg.output_dir, exist_ok=True)
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print("Saved model to:", cfg.output_dir)


## Deployment (Gradio)
The demo loads the saved model directory and predicts a topic for a given headline/article text.


In [ ]:
import gradio as gr

def load_classifier(model_dir: str):
    if not os.path.isdir(model_dir):
        raise FileNotFoundError(
            f"Model directory '{model_dir}' not found. Run training cells first to create it."
        )
    return pipeline(
        task="text-classification",
        model=model_dir,
        tokenizer=model_dir,
        return_all_scores=True,
        device=0 if torch.cuda.is_available() else -1,
    )

clf = load_classifier(cfg.output_dir)

def predict_topic(text: str):
    if not text or not text.strip():
        return {"error": "Please enter some text."}
    try:
        scores = clf(text.strip())[0]
        scores_sorted = sorted(scores, key=lambda d: d["score"], reverse=True)
        best = scores_sorted[0]
        return {
            "predicted_label": best["label"],
            "confidence": float(best["score"]),
            "all_scores": [
                {"label": s["label"], "score": float(s["score"])} for s in scores_sorted
            ],
        }
    except Exception as e:
        return {"error": str(e)}

demo = gr.Interface(
    fn=predict_topic,
    inputs=gr.Textbox(lines=2, label="Headline / Article"),
    outputs=gr.JSON(label="Prediction"),
    title="AG News Topic Classifier (BERT)",
    description=(
        "Fine-tuned bert-base-uncased on AG News. "
        "Defaults to a small subset so it runs quickly on free resources."
    ),
    examples=[
        "Apple unveils new iPhone as shares climb",
        "Local team wins championship after dramatic comeback",
        "Government passes new economic reform bill",
        "Scientists discover promising new treatment approach",
    ],
)

demo.launch()
